**This system leverages representations from RAVE. Learn more about that project here:** 
https://github.com/acids-ircam/RAVE

**Download more generative models from:** 
https://acids-ircam.github.io/rave_models_download

**Train your own model using:** 
https://colab.research.google.com/drive/1ih-gv1iHEZNuGhHPvCHrleLNXvooQMvI?usp=sharing (thanks to https://github.com/moiseshorta !)

**Giving Feedback** 
https://forms.office.com/Pages/DesignPageV2.aspx?prevorigin=Marketing&origin=NeoPortalPage&subpage=design&id=kfCdVhOw40CG7r2cueJYFFaGvjbF1INKikqB9y_N2gRUODBTNTJKM09HVjlFQjFRUlQ5U05QUVQ5MS4u&topview=Prefill

In [21]:
# In[0] # Load Utility Functions:
from load_generative_model import Model
from IPython.display import Audio, display
# from gui import interface
import librosa as li
import torch


import numpy as np
import os
from pathlib import Path
import IPython.display as ipd
from ipywidgets import interact, IntSlider
import matplotlib.pyplot as plt


In [22]:
# Working with trajectories in latent audio models

# In[1] # Pick Model:
model_name: str = 'percussion'
model_location:str = 'generative_models/'+model_name+'.ts'

control_model_location = 'control_models/vae_scripted_model.ts'
model = Model([model_location])
sr: int =44100

In [23]:
import os
import random

# In[2] # Pick Audio Samples:
# Get all sound files from the 'sounds' folder
sound_files = [f for f in os.listdir('sounds') if f.endswith(('.wav', '.aif', '.mp3', '.ogg'))]

# Select n random sounds
n = 8
selected_sounds = random.sample(sound_files, min(n, len(sound_files)))
# print(f"Selected sounds:")

# for sound in selected_sounds:
#     print(f"- {sound}")
#     audio_location = os.path.join('sounds', sound)
#     audio, sr = li.load(audio_location,sr=44100)
#     display(Audio(audio, rate=sr))

encodings = [model.encode(li.load(audio_location,sr=44100)[0]) for audio_location in [os.path.join('sounds', sound) for sound in selected_sounds]]
# for i, enc in enumerate(encodings):
#     print(f"Encoding {i} shape: {enc[0].shape}")
#     audio_recon = model.decode(enc[0])
#     display(Audio(audio_recon, rate=sr))




In [24]:
# Compute timbre
# Compute spectral centroid for each audio file
audio_features = []

for sound_file in [os.path.join('sounds', sound) for sound in selected_sounds]:
    try:
        # Load audio file
        y, sr = li.load(sound_file, sr=None)
        
        # Compute spectral centroid
        spectral_centroid = li.feature.spectral_centroid(y=y, sr=sr)[0]
        
        # Take the mean spectral centroid across time
        mean_centroid = np.mean(spectral_centroid)
        
        audio_features.append({
            'file': sound_file,
            'filename': Path(sound_file).name,
            'spectral_centroid': mean_centroid,
            'y': y,
            'sr': sr
        })
        
        print(f"Processed: {Path(sound_file).name} - Centroid: {mean_centroid:.2f} Hz")
    except Exception as e:
        print(f"Error processing {sound_file}: {e}")

print(f"\nSuccessfully processed {len(audio_features)} files.")



Processed: Guiro C78 Short.aif - Centroid: 11897.14 Hz
Processed: Clave C78.aif - Centroid: 10992.24 Hz
Processed: Metallic C78.aif - Centroid: 9849.78 Hz
Processed: Bongo C78 Hi.aif - Centroid: 3059.29 Hz
Processed: Guiro C78 Long.aif - Centroid: 11861.35 Hz
Processed: Rim C78.aif - Centroid: 8100.06 Hz
Processed: Guiro C78 Short b.aif - Centroid: 12239.48 Hz
Processed: Cowbell C78.aif - Centroid: 2532.93 Hz

Successfully processed 8 files.


/Users/ash/Documents/GitHub/Timbre-Slider/.venv/lib/python3.14/site-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=1661
  warnings.warn(


In [25]:
# Sort by timbre
# Sort by spectral centroid
sorted_audio = sorted(audio_features, key=lambda x: x['spectral_centroid'])
sorted_encodings = [encodings[selected_sounds.index(audio_info['filename'])] for audio_info in sorted_audio]

# Display the sorted list
print("Sounds sorted by spectral centroid (dark → bright):\n")
for i, audio_info in enumerate(sorted_audio, 1):
    print(f"{i:2d}. {audio_info['filename']:40s} | Centroid: {audio_info['spectral_centroid']:8.2f} Hz | Encoding shape: {sorted_encodings[i-1][0].shape}")

Sounds sorted by spectral centroid (dark → bright):

 1. Cowbell C78.aif                          | Centroid:  2532.93 Hz | Encoding shape: (1, 4, 2)
 2. Bongo C78 Hi.aif                         | Centroid:  3059.29 Hz | Encoding shape: (1, 4, 3)
 3. Rim C78.aif                              | Centroid:  8100.06 Hz | Encoding shape: (1, 4, 1)
 4. Metallic C78.aif                         | Centroid:  9849.78 Hz | Encoding shape: (1, 4, 3)
 5. Clave C78.aif                            | Centroid: 10992.24 Hz | Encoding shape: (1, 4, 2)
 6. Guiro C78 Long.aif                       | Centroid: 11861.35 Hz | Encoding shape: (1, 4, 40)
 7. Guiro C78 Short.aif                      | Centroid: 11897.14 Hz | Encoding shape: (1, 4, 6)
 8. Guiro C78 Short b.aif                    | Centroid: 12239.48 Hz | Encoding shape: (1, 4, 6)


In [ ]:
# print('encoding shapes:', [enc[0].shape for enc in encodings])


def resample_encodings(encodings, length:int=None):
    if length is None:
        max_len = max(enc[0].shape[-1] for enc in encodings)
    else:
        max_len = length
    resampled_encodings = [torch.nn.functional.interpolate(torch.from_numpy(enc[0]), size=max_len).numpy() for enc in encodings]
    return resampled_encodings



In [ ]:
# Interpolate across all encodings with two sliders:
# 1. Interpolation position (0 to len(encodings)-1, float)
# 2. Length to resample all encodings to (int)

import numpy as np
from ipywidgets import FloatSlider, IntSlider, interact, VBox, Label, fixed
from ipywidgets import Dropdown

# Default resample length
default_length = 40
encodings = sorted_encodings

# Slider for interpolation position
interp_slider = FloatSlider(min=0, max=len(encodings)-1, step=0.01, value=0, description='Interpolate')

# Slider for encoding length
length_slider = IntSlider(min=1, max=512, step=1, value=default_length, description='Length')

# Resample all encodings to the selected length
def resample_encodings(encodings, length:int=None):
    if length is None:
        max_len = max(enc[0].shape[-1] for enc in encodings)
    else:
        max_len = length
    resampled_encodings = [torch.nn.functional.interpolate(torch.from_numpy(enc[0]), size=max_len).numpy() for enc in encodings]
    return resampled_encodings

def interpolate_latents(enc_left, enc_right, alpha):
    return (1-alpha)*enc_left + alpha*enc_right

# Interpolation function: smoothly morph through all encodings
def interpolate_and_play(x, length, encodings, labels):
    resampled_encodings = resample_encodings(encodings, length)
    # print("shapes:", [enc.shape for enc in resampled_encodings])
    left_index = int(np.floor(x))
    right_index = max(left_index+1, len(resampled_encodings)-1)
    alpha = x - np.floor(x)

    interp = interpolate_latents(resampled_encodings[left_index], resampled_encodings[right_index], alpha)
    audio_interp = model.decode(interp)
    display(Audio(audio_interp, rate=sr))
    print(f"Interpolating between {selected_sounds[left_index]} and {selected_sounds[right_index]} (alpha={alpha:.2f}), length={length}")

    # Determine number of latent dimensions
    num_dims = interp.shape[1]

    # Dropdown to select dimension

    dim_selector = Dropdown(
        options=[(f"Dimension {i}", i) for i in range(num_dims)],
        value=0,
        description='Latent Dimension:',
    )

    def plot_latent(dim):
        plt.figure(figsize=(10, 4))
        plt.plot(interp[0, dim], label='Interpolated', color='black', linewidth=2)
        plt.plot(resampled_encodings[left_index][0, dim], label=f'Left ({labels[left_index]})', color='blue', linestyle='--')
        plt.plot(resampled_encodings[right_index][0, dim], label=f'Right ({labels[right_index]})', color='red', linestyle='--')
        plt.xlabel('Time')
        plt.ylabel('Value')
        plt.title(f'Latent Dimension {dim}')
        plt.legend()
        plt.tight_layout()
        plt.show()

    display(dim_selector)
    interact(plot_latent, dim=dim_selector)

    # Plot spectrograms for left, right, and interpolated audio
    def plot_spectrograms():
        fig, axs = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
        titles = [
            f"Left: {labels[left_index]}",
            f"Right: {labels[right_index]}",
            "Interpolated"
        ]
        audios = [
            model.decode(resampled_encodings[left_index]),
            model.decode(resampled_encodings[right_index]),
            audio_interp
        ]
        for i, (audio_data, ax, title) in enumerate(zip(audios, axs, titles)):
            S = np.abs(np.fft.rfft(audio_data))
            ax.plot(S)
            ax.set_title(title)
            ax.set_ylabel("Magnitude")
        axs[-1].set_xlabel("Frequency Bin")
        plt.tight_layout()
        plt.show()

        
    plot_spectrograms()

    


# print('Use the sliders to interpolate between sounds and set encoding length.')


In [43]:
print('Use the sliders to interpolate between sounds and set encoding length.')
# print(f"Options: {[(selected_sounds[x], encodings[x][0].shape) for x in range(len(encodings))]}")
interact(interpolate_and_play, x=interp_slider, length=length_slider, encodings=fixed(encodings), labels=fixed(selected_sounds))

Use the sliders to interpolate between sounds and set encoding length.


interactive(children=(FloatSlider(value=0.0, description='Interpolate', max=7.0, step=0.01), IntSlider(value=4…

<function __main__.interpolate_and_play(x, length, encodings, labels)>